In [1]:
def compare_dict_arrays(a, b, diff_fn, threshold):
    """
    Compare two arrays of dicts (a and b), returning an array with values:
    - None if a[i] == b[j] (diff == 0)
    - "edited" if 0 < diff <= threshold
    - "added" if diff > threshold or b is exhausted

    Args:
        a (list of dict): First array
        b (list of dict): Second array
        diff_fn (callable): Function that returns a numeric difference between two dicts
        threshold (float): Threshold to consider a difference as "edited"

    Returns:
        list: Array with values None, "edited", or "added" (same length as `a`)
    """
    result = []
    i, j = 0, 0

    while i < len(a):
        if j >= len(b):
            # No more items in b, all remaining in a are "added"
            result.append("added")
            i += 1
            continue

        delta = diff_fn(a[i], b[j])
        if delta == 0:
            result.append(None)
            i += 1
            j += 1
        elif delta <= threshold:
            result.append("edited")
            i += 1
            j += 1
        else:
            result.append("added")
            i += 1

    return result


In [2]:
def simple_diff(d1, d2):
    return sum(1 for k in d1 if d1.get(k) != d2.get(k))

a = [{'id': 1, 'val': 'A'}, {'id': 2, 'val': 'B'}, {'id': 3, 'val': 'X'}]
b = [{'id': 1, 'val': 'A'}, {'id': 2, 'val': 'C'}]

compare_dict_arrays(a, b, diff_fn=simple_diff, threshold=1)
# Output: [None, 'edited', 'added']


[None, 'edited', 'added']

In [7]:
def compare_dict_arrays(a, b, diff_fn, threshold):
    """
    Two-pass comparison of arrays of dicts.
    Pass 1: Match exact entries (diff == 0)
    Pass 2: On unmatched segments, detect edits or additions

    Returns a list the same length as `a`, with values:
      - None for exact match
      - "edited" for near matches
      - "added" for new items in a
    """
    result = [None] * len(a)
    matched_b = [False] * len(b)

    # Pass 1: Exact matches
    for i in range(len(a)):
        if i < len(b) and diff_fn(a[i], b[i]) == 0:
            result[i] = "equal"
            matched_b[i] = True

    # Pass 2: Edits and adds for unmatched segments
    i, j = 0, 0
    while i < len(a):
        if result[i] == "equal":
            i += 1
            j += 1
            continue

        # Advance j to next unmatched b[j]
        while j < len(b) and matched_b[j]:
            j += 1

        if j >= len(b):
            result[i] = "added"
            i += 1
            continue

        delta = diff_fn(a[i], b[j])
        if delta <= threshold:
            result[i] = "edited"
            matched_b[j] = True
            i += 1
            j += 1
        else:
            result[i] = "added"
            i += 1

    return result


In [8]:
compare_dict_arrays(a, b, diff_fn=simple_diff, threshold=1)

['equal', 'edited', 'added']

In [9]:
import difflib

def edit_distance(a, b):
    matcher = difflib.SequenceMatcher(None, a, b)
    return int(round((1 - matcher.ratio()) * max(len(a), len(b))))


In [15]:
edit_distance("foobar57", "fgoobaz159")

3